In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import os
import glob

import sys
sys.path.insert(1, './src')

### Load Diffusion pipeline

In [2]:
from utils_maps import *

exp_id = "2024-07-25_14-27-03_MLBriefs24_2_conditional_vanilla_downsample_mask"
exp_id = "2024-07-24_22-37-54_MLBriefs24_1_conditional_vanilla_encode_mask"
exp_id = "2024-07-26_04-40-24_MLBriefs24_0_unconditional"
exp_id = "2024-08-01_10-12-17_MLBriefs24_4_conditional_CA_downsampledmask"
#exp_id = "2024-08-02_14-35-03_MLBriefs24_5_conditional_CA_encodedmask"
model_path = f"/media/share/Datasets/diffusers_out/{exp_id}"

pipeline = load_pipeline(model_path)

Loading Diffusion pipeline from:
    - /media/share/Datasets/diffusers_out/2024-08-01_10-12-17_MLBriefs24_4_conditional_CA_downsampledmask

VQ-VAE loaded
U-Net model loaded
Diffusion pipeline is ready



/home/ubuntu/anaconda3/envs/diffusion/lib/python3.9/site-packages/diffusers/configuration_utils.py:244: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a scheduler, please use <class 'diffusers.schedulers.scheduling_ddpm.DDPMScheduler'>.from_pretrained(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)


### Generate N images

In [10]:
from PIL import Image
from tqdm import tqdm
import json
from datasets import load_dataset
from utils_maps import *

def save_pipeline_output_to_disk(output, batch_idx, batch_size, out_dir):
    images, cond_images = output
    os.makedirs(out_dir + "/ims", exist_ok=True)
    os.makedirs(out_dir + "/cond_ims", exist_ok=True)
    for idx in range(min(batch_size, images.shape[0])):
        local_d = {}
        global_img_idx = batch_idx * batch_size + idx
        img = Image.fromarray((images[idx]*255).astype(np.uint8))
        img.save(out_dir + f"/ims/{global_img_idx:05d}.png")
        if cond_images is not None:
            #cond_images[idx] = (cond_images[idx] * 0.5) + 0.5
            img = Image.fromarray((cond_images[idx]*255).astype(np.uint8))
            img.save(out_dir + f"/cond_ims/{global_img_idx:05d}.png")

def generate_N_images(pipeline, out_dir, n_images, batch_size=16, inference_steps=None, val_dataset=None):
    pipeline.nodule_attributes = True
    generator = torch.Generator(device=pipeline.device).manual_seed(0)
    if inference_steps is None:
        inference_steps = pipeline.scheduler.config.num_train_timesteps

    im_size = pipeline.vae.sample_size
    for batch_idx in tqdm(range(n_images//batch_size)):
        # prepare condition images
        if val_dataset is not None:
            sample_indices = np.arange(batch_idx*batch_size, (batch_idx+1)*batch_size)
            len_val = len(val_dataset)
            sample_indices[sample_indices >= len_val] = sample_indices[sample_indices >= len_val] % len_val
            input_condition_imgs = [val_dataset[int(i)]["condition"] for i in sample_indices]
            input_condition_imgs = torch.stack(input_condition_imgs)
        else:
            input_condition_imgs = None
        # run pipeline in inference (sample random noise and denoise)
        output = pipeline(
            input_condition_imgs=None if pipeline.args.unconditional else input_condition_imgs,
            generator=generator,
            batch_size=batch_size,
            num_inference_steps=inference_steps,
            output_type="numpy",
            return_dict=False
        )
        # save output images
        save_pipeline_output_to_disk(output, batch_idx, batch_size, out_dir)
        
    if n_images%batch_size > 0:
        batch_idx = n_images//batch_size
        batch_idx = batch_idx + 1 if batch_idx > 0 else 0
        # prepare condition images
        if val_dataset is not None:
            sample_indices = np.arange(batch_idx*batch_size, (batch_idx+1)*batch_size)
            len_val = len(val_dataset)
            sample_indices[sample_indices >= len_val] = sample_indices[sample_indices >= len_val] % len_val
            input_condition_imgs = [val_dataset[int(i)]["condition"] for i in sample_indices]
            input_condition_imgs = torch.stack(input_condition_imgs)
            input_condition_imgs = input_condition_imgs[:n_images%batch_size]
        else:
            input_condition_imgs = None
        # run pipeline in inference (sample random noise and denoise)
        output = pipeline(
            input_condition_imgs=None if pipeline.args.unconditional else input_condition_imgs,
            generator=generator,
            batch_size=n_images%batch_size,
            num_inference_steps=inference_steps,
            output_type="numpy",
            return_dict=False
        )
        # save output images
        save_pipeline_output_to_disk(output, batch_idx, batch_size, out_dir)


out_dir = model_path + "/fid_synthetic2"
n_images = 2048
bsz = 16

val_dataset = load_dataset(pipeline.args.val_dataset_name,
    pipeline.args.dataset_config_name,
    cache_dir=pipeline.args.cache_dir,
    split="train",
    )
val_dataset.set_transform(parse_maps_val)

generate_N_images(pipeline, out_dir, n_images, batch_size=bsz, inference_steps=1000, val_dataset=val_dataset)

print("DONE")

Resolving data files:   0%|          | 0/1098 [00:00<?, ?it/s]

/tmp/ipykernel_2606223/336401626.py:28: FutureWarning: Accessing config attribute `sample_size` directly via 'VQModel' object attribute is deprecated. Please access 'sample_size' over 'VQModel's config object instead, e.g. 'unet.config.sample_size'.
  im_size = pipeline.vae.sample_size
  0%|                                                                                                                                                                  | 0/1 [00:00<?, ?it/s]

tensor([[[0.3804, 0.3804, 0.3961,  ..., 0.3725, 0.3647, 0.3647],
         [0.3882, 0.3882, 0.3961,  ..., 0.3725, 0.3647, 0.3647],
         [0.3882, 0.3882, 0.3882,  ..., 0.3647, 0.3725, 0.3647],
         ...,
         [0.3882, 0.3882, 0.3882,  ..., 0.7490, 0.7490, 0.7490],
         [0.3882, 0.3882, 0.3882,  ..., 0.7490, 0.7490, 0.7490],
         [0.3882, 0.3882, 0.3882,  ..., 0.7490, 0.7490, 0.7490]],

        [[0.6235, 0.6235, 0.6392,  ..., 0.6157, 0.6078, 0.6078],
         [0.6314, 0.6314, 0.6392,  ..., 0.6157, 0.6078, 0.6000],
         [0.6314, 0.6314, 0.6314,  ..., 0.6157, 0.6000, 0.5922],
         ...,
         [0.6314, 0.6314, 0.6314,  ..., 0.7098, 0.7098, 0.7098],
         [0.6314, 0.6314, 0.6314,  ..., 0.7098, 0.7098, 0.7098],
         [0.6314, 0.6314, 0.6314,  ..., 0.7098, 0.7098, 0.7098]],

        [[0.9765, 0.9765, 1.0000,  ..., 0.9765, 0.9765, 0.9765],
         [0.9922, 0.9922, 1.0000,  ..., 0.9843, 0.9843, 0.9843],
         [0.9922, 0.9922, 0.9922,  ..., 1.0000, 1.0000, 1.

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.24s/it]


UnboundLocalError: local variable 'ue' referenced before assignment

In [12]:
# rewrite stuff

exp_id = "2024-07-24_22-37-54_MLBriefs24_1_conditional_vanilla_encode_mask"
exp_id = "2024-07-25_14-27-03_MLBriefs24_2_conditional_vanilla_downsample_mask"
exp_id = "2024-08-02_14-35-03_MLBriefs24_5_conditional_CA_encodedmask"
exp_id = "2024-08-01_10-12-17_MLBriefs24_4_conditional_CA_downsampledmask"

out_dir = model_path + "/fid_synthetic"

for img_p in tqdm(sorted(glob.glob(out_dir + "/cond_ims/*.png"))):
    img = np.array(Image.open(img_p)) / 255.
    img = (img * 0.5) + 0.5
    img = (img * 255).astype(np.uint8)
    out_p = img_p.replace("fid_synthetic", "fid_synthetic_2")
    os.makedirs(os.path.dirname(out_p), exist_ok=True)
    Image.fromarray(img).save(out_p)


  0%|▌                                                                                                                                                      | 8/2048 [00:00<02:39, 12.78it/s]


KeyboardInterrupt: 

### Compute FID

In [19]:
from tqdm import tqdm
from PIL import Image
import torch
from torchvision import transforms


exp_id = "2024-07-26_04-40-24_MLBriefs24_0_unconditional"
exp_id = "2024-07-24_22-37-54_MLBriefs24_1_conditional_vanilla_encode_mask"
exp_id = "2024-07-25_14-27-03_MLBriefs24_2_conditional_vanilla_downsample_mask"
exp_id = "2024-08-02_14-35-03_MLBriefs24_5_conditional_CA_encodedmask"
exp_id = "2024-08-01_10-12-17_MLBriefs24_4_conditional_CA_downsampledmask"
model_path = f"/media/share/Datasets/diffusers_out/{exp_id}"

fake_images_dir = f"{model_path}/fid_synthetic/ims"
fake_images_dir = "/home/ubuntu/projects/phase-iv-ai/maps/val_ims"
real_images_dir = "/home/ubuntu/projects/phase-iv-ai/maps/val_ims"
n_images = 2048

def read_image_dir_for_FID_computation(input_dir, resolution=256, n_images=None,
                                       img_extension=".png", split="train"):
    augmentations = transforms.Compose(
        [
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            #transforms.ToTensor(),
            #transforms.Normalize([0.5], [0.5]),
        ]
    )
    
    assert os.path.exists(input_dir)
    img_paths = sorted(glob.glob(os.path.join(input_dir, f"*{img_extension}")))
    images = []
    for p in tqdm(img_paths):
        img = np.array(Image.open(p))
        if split == "train":
            w = img.shape[1]
            img = img[:, :w//2, :]
        img = torch.from_numpy(img).permute(2, 0, 1).type(torch.float32) / 255.
        if img.shape[1] != resolution:
            img = augmentations(img)
        images.append(img)
    images = torch.stack(images)
    # we need at least 2048 images to compute the FID
    #https://github.com/mseitzer/pytorch-fid/issues/13
    if (n_images is not None) and (images.shape[0] < n_images):
        diff = n_images - images.shape[0]
        images = torch.cat([images, images[:diff]], dim=0)
    return images

# read real images
real_images = read_image_dir_for_FID_computation(real_images_dir, img_extension=".png", split="real")
print(f"real images loaded. shape is {real_images.shape}")

fake_images = read_image_dir_for_FID_computation(fake_images_dir, img_extension=".png", split="fake")
print(f"fake images loaded. shape is {fake_images.shape}")


from fid import FID

# requires !pip install scipy==1.9.1
fid = FID(real_images.cuda(), device="cuda")
fid_score = fid.calculate_FID(fake_images.cuda())
print(fid_score)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1098/1098 [00:03<00:00, 327.03it/s]


real images loaded. shape is torch.Size([1098, 3, 256, 256])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1098/1098 [00:03<00:00, 332.47it/s]


fake images loaded. shape is torch.Size([1098, 3, 256, 256])
[FID] Computing activations from 1098 samples
-3.487417836822715e-05


### Save images to disk

In [28]:
"""
for idx, image in enumerate(fake_images):
    im = image.permute(1,2,0).cpu().numpy()
    Image.fromarray((im*255).astype(np.uint8)).save(f"/home/ubuntu/projects/phase-iv-ai/maps/val_ims/{idx+1}.png")
    
for idx, image in enumerate(real_images):
    im = image.permute(1,2,0).cpu().numpy()
    Image.fromarray((im*255).astype(np.uint8)).save(f"/home/ubuntu/projects/phase-iv-ai/maps/train_ims/{idx+1}.png")
    
print("ok")
"""

ok
